# HAM10000 Skin Lesion Classification (Imbalanced Multi-class CNN)
**Applied Machine Learning Advanced — Final Project**

## Summary
We tackle multi-class classification on the HAM10000 dermatoscopic dataset (10,015 images, 7 classes).
Key challenges include strong class imbalance (melanocytic nevi dominate) and the need for robust augmentation.

We implement:
- A strong **Transfer Learning** baseline (EfficientNet-B0).
- Two imbalance strategies: **Class-Weighted Cross Entropy** and **Focal Loss**.
- Proper evaluation: **Macro-F1**, per-class Recall, Confusion Matrix.
- **Grad-CAM** for interpretability.

---


In [5]:
import os, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from PIL import Image, ImageFile, UnidentifiedImageError
ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

DATA_DIR = Path("/content/data")
META_PATH = DATA_DIR / "HAM10000_metadata.csv"
IMG_DIR_1 = DATA_DIR / "HAM10000_images_part_1"
IMG_DIR_2 = DATA_DIR / "HAM10000_images_part_2"

print("META exists:", META_PATH.exists())
print("IMG1 exists:", IMG_DIR_1.exists())
print("IMG2 exists:", IMG_DIR_2.exists())


Device: cpu
META exists: False
IMG1 exists: False
IMG2 exists: False


In [6]:
assert META_PATH.exists(), "HAM10000_metadata.csv not found in /content/data"

df = pd.read_csv(META_PATH)
print("Metadata shape:", df.shape)

def get_image_path(image_id: str):
    p1 = IMG_DIR_1 / f"{image_id}.jpg"
    if p1.exists(): return str(p1)
    p2 = IMG_DIR_2 / f"{image_id}.jpg"
    if p2.exists(): return str(p2)
    return None

df["path"] = df["image_id"].apply(get_image_path)
df = df.dropna(subset=["path"]).reset_index(drop=True)
print("After path dropna:", df.shape)
df.head()


AssertionError: HAM10000_metadata.csv not found in /content/data

In [ ]:
classes = sorted(df["dx"].unique())
label2id = {c:i for i,c in enumerate(classes)}
id2label = {i:c for c,i in label2id.items()}
df["label"] = df["dx"].map(label2id)

print("Classes:", classes)
df["dx"].value_counts()


Classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


,count
dx,
nv,6230
bkl,997
mel,924
bcc,485
akiec,321
vasc,131
df,102


In [ ]:
bad = []
for p in df["path"].tolist():
    try:
        with Image.open(p) as im:
            im.verify()
    except Exception:
        bad.append(p)

print("Corrupted images:", len(bad))


Corrupted images: 0


In [ ]:
if len(bad) > 0:
    bad_set = set(bad)
    df = df[~df["path"].isin(bad_set)].reset_index(drop=True)
print("Clean df:", df.shape)


Clean df: (9190, 9)


In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df["label"], random_state=SEED
)
train_df, val_df = train_test_split(
    train_df, test_size=0.15, stratify=train_df["label"], random_state=SEED
)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)


Train: (6639, 9) Val: (1172, 9) Test: (1379, 9)


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

weights = EfficientNet_B0_Weights.DEFAULT
mean, std = weights.transforms().mean, weights.transforms().std

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

class HAMDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.df.loc[idx, "path"]
        label = int(self.df.loc[idx, "label"])
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        return img, label

train_ds = HAMDataset(train_df, train_tfms)
val_ds   = HAMDataset(val_df, eval_tfms)
test_ds  = HAMDataset(test_df, eval_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("Loaders ready ")


Loaders ready 


In [ ]:
train_counts = train_df["label"].value_counts().sort_index().values
class_weights = train_counts.sum() / (len(classes) * train_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Train counts:", train_counts)
print("Class weights:", class_weights)


Train counts: [ 232  350  721   74  667 4501   94]
Class weights: tensor([ 4.0881,  2.7098,  1.3154, 12.8166,  1.4219,  0.2107, 10.0897])


In [ ]:
def build_model(num_classes):
    m = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    in_features = m.classifier[1].in_features
    m.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes),
    )
    return m

model = build_model(len(classes)).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

print("Model ready")


Model ready


In [ ]:
from tqdm import tqdm
scaler = torch.cuda.amp.GradScaler(enabled=(device=="cuda"))

def train_one_epoch(model, loader):
    model.train()
    total_loss, n = 0.0, 0
    for x, y in tqdm(loader, leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device=="cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * x.size(0)
        n += x.size(0)
    return total_loss / n

@torch.no_grad()
def eval_macro_f1(model, loader):
    model.eval()
    y_true, y_pred = [], []
    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        pred = logits.argmax(dim=1).cpu().numpy()
        y_true.extend(y.numpy())
        y_pred.extend(pred)
    return f1_score(y_true, y_pred, average="macro")

EPOCHS = 5  # start small to verify everything runs

best_f1 = -1
for epoch in range(1, EPOCHS+1):
    tr_loss = train_one_epoch(model, train_loader)
    val_f1  = eval_macro_f1(model, val_loader)
    print(f"Epoch {epoch:02d} | train_loss={tr_loss:.4f} | val_macroF1={val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "/content/outputs/models/best_effb0.pt")

print("Best Val Macro-F1:", best_f1)


/tmp/ipython-input-3463304593.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device=="cuda"))
  0%|          | 0/208 [00:00<?, ?it/s]/tmp/ipython-input-3463304593.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device=="cuda")):


Epoch 01 | train_loss=0.6716 | val_macroF1=0.6356


  0%|          | 0/208 [00:00<?, ?it/s]/tmp/ipython-input-3463304593.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device=="cuda")):
 66%|██████▌   | 137/208 [23:13<11:32,  9.76s/it]